# Simple RNN: Hotel Review Sentiment Classification
**SE4050 Deep Learning 2026 · Individual Model Contribution: Simple RNN**

This master all-in-one notebook executes the end-to-end sentiment classification pipeline:
1. Environment & libraries verification
2. Dataset audit (hygiene, encoding, duplicates, outliers)
3. Fixed 80/10/10 stratified split
4. Exploratory data analysis (class balance, review length, distinctive tokens, length baseline)
5. Text cleaning, tokenization & vocabulary construction (train set only)
6. Pre-padding & sequence formatting
7. Balanced class weights
8. Simple RNN architecture definition & exact parameter counting (330,369 parameters)
9. Model training with Adam and EarlyStopping
10. Learning curves visualization (Accuracy and Loss)
11. Final evaluation on the unseen test set (Bootstrap 95% CI, confusion matrix, ROC curve)
12. Comprehensive error analysis (FP vs FN, confidence, vanishing gradient effects)
13. Multi-seed robustness validation across 6 seeds
14. Handover summary for group comparison
15. Pre-loaded battery of 8 challenging custom reviews
16. Interactive review classification interface ("Try Your Own Review")

## 1. Setup, Environment & Dataset Location

In [ ]:
# Imports and environment setup
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers
IN_COLAB = "google.colab" in sys.modules

# Locate root directory and results folder
here = pathlib.Path.cwd().resolve()
RNN_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "simple_rnn").is_dir()), here)
if (RNN_ROOT / "simple_rnn").is_dir():
    RNN_ROOT = RNN_ROOT / "simple_rnn"
RESULTS = RNN_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# Locate dataset file with interactive fallback for Google Colab
DATA_NAME = "deceptive-opinion.csv"
search_in = [RNN_ROOT, RNN_ROOT / "dataset", RNN_ROOT.parent / "dataset", here, here / "dataset",
             here.parent / "dataset", pathlib.Path("/content")]
DATA_PATH = next((d / DATA_NAME for d in search_in if (d / DATA_NAME).exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose {DATA_NAME} from your computer...")
    uploaded = files.upload()
    DATA_PATH = here / next(iter(uploaded))

if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Please ensure dataset/{DATA_NAME} is placed in the workspace "
                            "or upload it when prompted in Google Colab.")

print("Dataset:", DATA_PATH)
print("Results folder:", RESULTS)

## 2. Group Settings & Hyperparameters (Locked Across Group)

In [ ]:
# Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN
SEED = 42

LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for test metrics

# Fixed 80/10/10 split (stratified on the label, seed 42)
VAL_IDS = {
    3, 4, 6, 7, 14, 24, 25, 27, 33, 38, 46, 50, 60, 66, 72, 74, 77, 85, 89, 92, 106, 110, 112, 136,
    144, 172, 211, 215, 224, 230, 238, 242, 248, 252, 288, 302, 311, 319, 336, 345, 368, 376, 385,
    393, 402, 404, 434, 440, 442, 465, 473, 510, 512, 515, 517, 524, 533, 536, 542, 549, 553, 558,
    579, 581, 594, 603, 624, 630, 638, 644, 645, 661, 663, 665, 699, 713, 729, 754, 760, 788, 801,
    813, 820, 821, 827, 832, 840, 864, 873, 893, 897, 900, 904, 907, 908, 923, 930, 936, 962, 963,
    989, 991, 1002, 1008, 1017, 1043, 1045, 1054, 1073, 1098, 1112, 1119, 1134, 1143, 1170, 1189,
    1191, 1201, 1239, 1258, 1262, 1263, 1264, 1270, 1280, 1295, 1308, 1309, 1314, 1328, 1344, 1350,
    1359, 1360, 1364, 1367, 1368, 1369, 1370, 1377, 1378, 1381, 1418, 1423, 1450, 1467, 1479, 1480,
    1490, 1505, 1506, 1522, 1547, 1551, 1552, 1573, 1578, 1588, 1593, 1594
}
TEST_IDS = {
    0, 2, 5, 9, 39, 45, 47, 55, 108, 117, 120, 125, 138, 139, 145, 171, 191, 193, 213, 222, 227,
    228, 232, 256, 258, 260, 272, 273, 276, 281, 299, 314, 321, 325, 349, 362, 364, 391, 409, 424,
    436, 445, 467, 469, 484, 502, 503, 505, 507, 523, 525, 531, 545, 547, 572, 573, 605, 606, 612,
    622, 623, 633, 669, 671, 681, 687, 691, 698, 718, 725, 731, 741, 752, 753, 765, 770, 771, 779,
    784, 785, 804, 814, 834, 835, 847, 852, 859, 866, 882, 887, 889, 901, 932, 940, 958, 961, 968,
    972, 993, 1003, 1007, 1019, 1020, 1033, 1044, 1046, 1055, 1072, 1076, 1079, 1091, 1117, 1141,
    1147, 1149, 1151, 1176, 1182, 1183, 1193, 1195, 1196, 1205, 1210, 1217, 1222, 1231, 1259, 1265,
    1275, 1278, 1279, 1288, 1293, 1296, 1302, 1306, 1312, 1324, 1357, 1365, 1366, 1396, 1404, 1416,
    1446, 1451, 1465, 1478, 1485, 1487, 1492, 1503, 1504, 1516, 1524, 1540, 1562, 1582, 1597
}
keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])

## 3. Data Pipeline & Helper Functions

In [ ]:
# Data pipeline helpers
def load_labelled_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    df["row_id"] = np.arange(len(df))
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df[LABEL_COLUMN].map(LABEL_MAP)
    assert df["label"].notna().all(), "unexpected label value"
    df["label"] = df["label"].astype(int)
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    return df[["row_id", "text", "label", "hotel", "deceptive", "source"]]

def assign_split(df: pd.DataFrame) -> np.ndarray:
    if not (VAL_IDS | TEST_IDS) <= set(df["row_id"]):
        raise ValueError("The fixed validation/test ids do not match this dataset.")
    return np.where(df["row_id"].isin(TEST_IDS), "test",
                    np.where(df["row_id"].isin(VAL_IDS), "val", "train"))

_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"(https?://\S+|www\.\S+)")
_CONTRACTIONS = [
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'ve\b", " have"), (r"'ll\b", " will"),
    (r"'d\b", " would"), (r"'m\b", " am"), (r"'s\b", ""),
]
_NON_LETTER = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")

def clean_text(text: str) -> str:
    text = text.lower().replace("’", "'")
    text = _HTML.sub(" ", text)
    text = _URL.sub(" ", text)
    for pattern, replacement in _CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = _NON_LETTER.sub(" ", text)
    return _SPACES.sub(" ", text).strip()

def build_vocab(train_token_lists, vocab_size: int = VOCAB_SIZE) -> dict:
    counts = Counter(tok for tokens in train_token_lists for tok in tokens)
    words = [w for w, _ in counts.most_common(vocab_size - 2)]
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab

def encode(tokens, vocab: dict) -> list:
    oov = vocab[OOV_TOKEN]
    return [vocab.get(t, oov) for t in tokens]

def decode(ids, vocab: dict, skip_pad: bool = True) -> list:
    inverse = {i: w for w, i in vocab.items()}
    return [inverse[int(i)] for i in ids if not (skip_pad and int(i) == 0)]

def pad(sequences, max_len: int = MAX_LEN) -> np.ndarray:
    out = np.zeros((len(sequences), max_len), dtype="int32")
    for i, seq in enumerate(sequences):
        seq = seq[:max_len] if TRUNCATING == "post" else seq[-max_len:]
        if PADDING == "pre":
            out[i, max_len - len(seq):] = seq
        else:
            out[i, :len(seq)] = seq
    return out

def compute_class_weights(y_train) -> dict:
    classes, counts = np.unique(y_train, return_counts=True)
    return {int(c): float(len(y_train) / (len(classes) * n)) for c, n in zip(classes, counts)}

@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    vocab: dict
    class_weight: dict
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame

def prepare_data() -> Data:
    df = load_labelled_data()
    df["split"] = assign_split(df)
    df["clean"] = df["text"].map(clean_text)
    df["tokens"] = df["clean"].str.split()

    parts = {name: g.reset_index(drop=True) for name, g in df.groupby("split")}
    vocab = build_vocab(parts["train"]["tokens"].tolist())

    def to_xy(part: pd.DataFrame):
        seqs = [encode(tokens, vocab) for tokens in part["tokens"]]
        return pad(seqs), part["label"].to_numpy(dtype="int32")

    X_train, y_train = to_xy(parts["train"])
    X_val, y_val = to_xy(parts["val"])
    X_test, y_test = to_xy(parts["test"])
    return Data(X_train, y_train, X_val, y_val, X_test, y_test, vocab,
                compute_class_weights(y_train), parts["train"], parts["val"], parts["test"])

In [ ]:
# Evaluation metrics
def compute_metrics(y_true, y_prob, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "negative_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def bootstrap_ci(y_true, y_prob, n_boot: int = 2000, seed: int = SEED) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    rng = np.random.default_rng(seed)
    out = {"accuracy": [], "f1": [], "roc_auc": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_metrics(y_true[idx], y_prob[idx])
        for k in out:
            out[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in out.items()}

def save_result(path, model_name: str, test_metrics: dict, *, training_time_sec: float,
                trainable_params: int, total_params: int, epochs_run: int, best_epoch: int,
                extra: dict = None) -> dict:
    row = {
        "model": model_name,
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"],
        "recall": test_metrics["recall"], "f1": test_metrics["f1"], "roc_auc": test_metrics["roc_auc"],
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params, "total_params": total_params,
        "epochs_run": epochs_run, "best_epoch": best_epoch,
        "seed": SEED, "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE,
        "embed_dim": EMBED_DIM, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "threshold": THRESHOLD,
        "confusion_matrix": {k: test_metrics[k] for k in ("tn", "fp", "fn", "tp")},
    }
    if extra:
        row.update(extra)
    with open(path, "w") as f:
        json.dump(row, f, indent=2)
    return row

In [ ]:
# Plot styling helpers
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
BLUE_RAMP = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#3987e5", "#0d366b"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 2,
})

def save_and_show(fig, path=None, show=True):
    fig.tight_layout()
    if path:
        fig.savefig(path)
    if show:
        plt.show()
    plt.close(fig)

def plot_curve(history: dict, metric: str, model_name: str, best_epoch=None, path=None, show=True):
    train, val = history[metric], history["val_" + metric]
    epochs = np.arange(1, len(train) + 1)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(epochs, train, color=BLUE, label="Training")
    ax.plot(epochs, val, color=ORANGE, label="Validation")
    for series, color, name in ((train, BLUE, "Training"), (val, ORANGE, "Validation")):
        ax.plot(epochs[-1], series[-1], "o", color=color, markersize=6,
                markeredgecolor="white", markeredgewidth=1.5)
        ax.annotate(name, (epochs[-1], series[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color=INK2, fontsize=10)
    if best_epoch:
        ax.axvline(best_epoch, color=INK2, linestyle=":", linewidth=1.2)
        ax.text(best_epoch, ax.get_ylim()[1], f" best epoch {best_epoch}", color=INK2,
                fontsize=9, va="top", ha="left")
    ax.set_xlim(0.7, epochs[-1] + 0.15 * max(epochs[-1], 6))
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"{model_name}: training vs validation {metric}", loc="left")
    ax.legend(loc="best")
    save_and_show(fig, path, show)

def plot_confusion_matrix(cm: np.ndarray, model_name: str, path=None, show=True):
    labels = ["Negative", "Positive"]
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(cm, cmap=BLUE_RAMP, vmin=0, vmax=cm.max())
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            dark = cm[i, j] > cm.max() * 0.55
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18,
                    fontweight="bold", color="white" if dark else INK)
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name}: confusion matrix (test)", loc="left")
    for s in ax.spines.values():
        s.set_visible(False)
    save_and_show(fig, path, show)

def plot_roc(y_true, y_prob, auc: float, model_name: str, threshold: float = 0.5, path=None, show=True):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5.2, 4.8))
    ax.plot([0, 1], [0, 1], color=INK2, linestyle="--", linewidth=1.2)
    ax.text(0.55, 0.47, "chance (AUC = 0.50)", color=INK2, fontsize=9, rotation=33)
    ax.plot(fpr, tpr, color=BLUE)
    k = np.argmin(np.abs(thr - threshold))
    ax.plot(fpr[k], tpr[k], "o", color=ORANGE, markersize=8, markeredgecolor="white", markeredgewidth=1.5)
    ax.annotate(f"threshold {threshold}", (fpr[k], tpr[k]), xytext=(10, -14),
                textcoords="offset points", color=INK2, fontsize=9)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{model_name}: ROC curve (test), AUC = {auc:.3f}", loc="left")
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    ax.set_aspect("equal")
    save_and_show(fig, path, show)

def plot_probability_hist(y_true, y_prob, model_name: str, threshold: float = 0.5, path=None, show=True):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob).ravel()
    bins = np.linspace(0, 1, 21)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.hist([y_prob[y_true == 1], y_prob[y_true == 0]], bins=bins, stacked=True,
            color=[BLUE, ORANGE], label=["True positive reviews", "True negative reviews"],
            edgecolor="white", linewidth=1.5)
    ax.axvline(threshold, color=INK2, linestyle=":", linewidth=1.2)
    ax.set_xlabel("Predicted probability of positive")
    ax.set_ylabel("Number of test reviews")
    ax.set_title(f"{model_name}: confidence on the test set", loc="left")
    ax.legend(loc="upper center")
    save_and_show(fig, path, show)

## 4. Step 1: Data Audit (Hygiene Checks & Outliers)

In [ ]:
raw = pd.read_csv(DATA_PATH)
text = raw["text"].astype(str).str.strip()
cleaned = text.map(clean_text)

print(f"Total raw rows: {len(raw):,} | columns: {list(raw.columns)}")
print("\nMissing values per column:\n", raw.isna().sum())
print("\nDuplicate review texts:", text.duplicated().sum())

tokens = cleaned.str.split().map(len)
q1, q3 = tokens.quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print(f"\nToken length: min={tokens.min()}, median={tokens.median():.0f}, max={tokens.max()}")
print(f"IQR outlier fences: lower={lower:.0f}, upper={upper:.0f} -> {int((tokens > upper).sum())} outliers above upper fence.")

fig, ax = plt.subplots(figsize=(7, 2.9))
groups = [tokens[raw["polarity"] == k] for k in ("positive", "negative")]
box_style = dict(widths=0.55, patch_artist=True,
                 flierprops=dict(marker="o", markersize=3, markerfacecolor=INK2, markeredgecolor="none", alpha=0.5),
                 medianprops=dict(color="white", linewidth=2), whiskerprops=dict(color=INK2), capprops=dict(color=INK2))
bp = ax.boxplot(groups, orientation="horizontal", **box_style)
for box, colour in zip(bp["boxes"], (BLUE, ORANGE)):
    box.set(facecolor=colour, edgecolor="none")
ax.set_yticks([1, 2], ["Positive", "Negative"])
ax.set_xlabel("Tokens per review (dots = outliers by the IQR rule)")
ax.set_title("Review length outliers", loc="left")
save_and_show(fig, RESULTS / "simple_rnn_audit_length_outliers.png")

## 5. Step 2: Dataset & Fixed Stratified Split

In [ ]:
data = prepare_data()
print("DATASET SPLIT SUMMARY:")
print(f"  Training set:   {len(data.y_train):,} reviews (positive: {(data.y_train == 1).sum()}, negative: {(data.y_train == 0).sum()})")
print(f"  Validation set: {len(data.y_val):,} reviews (positive: {(data.y_val == 1).sum()}, negative: {(data.y_val == 0).sum()})")
print(f"  Test set:       {len(data.y_test):,} reviews (positive: {(data.y_test == 1).sum()}, negative: {(data.y_test == 0).sum()})")
print(f"  Total:          {len(data.y_train) + len(data.y_val) + len(data.y_test):,} reviews")
assert len(VAL_IDS & TEST_IDS) == 0, "Disjoint sets assertion failed!"
print("Split verified: Exact 80/10/10 stratified split with zero test leakage.")

## 6. Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class Balance Plot
fig, ax = plt.subplots(figsize=(5, 3.5))
counts = data.train_df["label"].value_counts().sort_index()
bars = ax.bar(["Negative (0)", "Positive (1)"], counts, color=[ORANGE, BLUE], width=0.55)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, yval + 10, f"{yval:,} ({yval/len(data.train_df):.1%})",
            ha="center", va="bottom", fontsize=10, color=INK)
ax.set_ylim(0, 800)
ax.set_ylabel("Number of reviews")
ax.set_title("Training set class balance", loc="left")
save_and_show(fig, RESULTS / "simple_rnn_eda_class_balance.png")

# Distinctive Vocabulary Analysis
train = data.train_df
pos_tokens = [w for toks in train[train.label == 1]["tokens"] for w in toks]
neg_tokens = [w for toks in train[train.label == 0]["tokens"] for w in toks]
pos_counts, neg_counts = Counter(pos_tokens), Counter(neg_tokens)
all_words = set(pos_counts) | set(neg_counts)
scores = {w: np.log((pos_counts[w] + 1) / len(pos_tokens)) - np.log((neg_counts[w] + 1) / len(neg_tokens))
          for w in all_words if pos_counts[w] + neg_counts[w] >= 15}

items = sorted(scores.items(), key=lambda x: x[1])[:10] + sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10][::-1]
words, vals = zip(*items)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(words, vals, color=[ORANGE if v < 0 else BLUE for v in vals], height=0.65)
ax.axvline(0, color=INK2, linestyle=":", linewidth=1)
ax.set_xlabel("Log Odds Ratio (Negative ← | → Positive)")
ax.set_title("Most distinctive sentiment words", loc="left")
save_and_show(fig, RESULTS / "simple_rnn_eda_distinctive_words.png")

# Review length heuristic check
train = train.assign(n_tokens=train["tokens"].map(len))
val = data.val_df.assign(n_tokens=data.val_df["tokens"].map(len))
best_thresh, best_acc = 0, 0
for t in range(50, 250, 5):
    acc = ((train["n_tokens"] <= t).astype(int) == train["label"]).mean()
    if acc > best_acc: best_acc, best_thresh = acc, t
val_acc = ((val["n_tokens"] <= best_thresh).astype(int) == val["label"]).mean()
print(f"Length-only heuristic baseline (<= {best_thresh} tokens -> positive): Val Accuracy = {val_acc:.1%}")

## 7. Step 4: Feature Engineering & Pre-Padding Rationale

In [ ]:
print(f"Vocabulary size: {len(data.vocab):,} tokens (PAD=0, OOV=1)")
lengths = data.train_df["tokens"].map(len)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(lengths, bins=40, color=BLUE, edgecolor="white", alpha=0.85)
ax.axvline(MAX_LEN, color=ORANGE, linestyle="--", linewidth=1.8, label=f"MAX_LEN = {MAX_LEN}")
ax.set_xlabel("Review length (tokens)")
ax.set_ylabel("Count")
ax.set_title(f"Review length distribution ({100 * (lengths <= MAX_LEN).mean():.1f}% within {MAX_LEN} tokens)", loc="left")
ax.legend()
save_and_show(fig, RESULTS / "simple_rnn_eda_review_length.png")

## 8. Step 5: Class Weights

In [ ]:
print("Computed Class Weights (training set):")
for c, w in data.class_weight.items():
    name = "Positive" if c == 1 else "Negative"
    print(f"  Class {c} ({name}): {w:.4f}")

## 9. Step 6: Simple RNN Model Architecture & Exact Parameter Count

```
Input (300 token ids)
  → Embedding(5000, 64)        320,000 params (5000 * 64)
  → SimpleRNN(64)                8,256 params (W_xh: 64*64 + W_hh: 64*64 + b: 64)
  → Dropout(0.5)                     0 params
  → Dense(32, ReLU)              2,080 params (64 * 32 + 32)
  → Dropout(0.3)                     0 params
  → Dense(1, Sigmoid)               33 params (32 * 1 + 1)
Total Parameters:              330,369 (100% trainable)
```

In [ ]:
def build_simple_rnn():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.SimpleRNN(64),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="simple_rnn_sentiment")

model = build_simple_rnn()
model.summary()

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

breakdown = {layer.name: n_params(layer.trainable_weights) for layer in model.layers if layer.trainable_weights}
print("\nTrainable parameter breakdown:", breakdown)
print(f"Total Parameters: {model.count_params():,}")
print(f"Embedding Share: {100 * breakdown['embedding'] / model.count_params():.2f}%")
print(f"Simple RNN Layer: {breakdown['simple_rnn']:,} params (Input kernel: 4,096, Recurrent kernel: 4,096, Bias: 64)")

## 10. Step 7: Model Compilation & Training

In [ ]:
def train_simple_rnn(seed, verbose=2):
    keras.utils.set_random_seed(seed)
    m = build_simple_rnn()
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="binary_crossentropy", metrics=["accuracy"])
    early_stop = keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE,
                                               restore_best_weights=True)
    start = time.perf_counter()
    hist = m.fit(data.X_train, data.y_train,
                 validation_data=(data.X_val, data.y_val),
                 epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                 class_weight=data.class_weight, callbacks=[early_stop], verbose=verbose)
    return m, hist.history, time.perf_counter() - start

def save_training_outputs(model, history, train_time, epochs_run, best_epoch):
    model.save(RESULTS / "simple_rnn_model.keras")
    (pd.DataFrame(history).rename_axis("epoch").reset_index().assign(epoch=lambda d: d.epoch + 1)
       .to_csv(RESULTS / "simple_rnn_history.csv", index=False))
    (RESULTS / "simple_rnn_training_info.json").write_text(json.dumps(
        {"train_time_sec": train_time, "epochs_run": epochs_run, "best_epoch": best_epoch}))

model, history, train_time = train_simple_rnn(SEED)
epochs_run = len(history["loss"])
best_epoch = int(np.argmin(history["val_loss"])) + 1
print(f"\nTraining complete in {train_time:.1f} s ({train_time / epochs_run:.1f} s/epoch)")
print(f"Total Epochs Run: {epochs_run} / {MAX_EPOCHS} | Best Epoch: {best_epoch}")

save_training_outputs(model, history, train_time, epochs_run, best_epoch)
print("Saved simple_rnn_model.keras, simple_rnn_history.csv and simple_rnn_training_info.json")

## 11. Step 8: Learning Curves (Loss & Accuracy)

In [ ]:
plot_curve(history, "accuracy", "Simple RNN", best_epoch=best_epoch,
           path=RESULTS / "simple_rnn_accuracy_curve.png")

plot_curve(history, "loss", "Simple RNN", best_epoch=best_epoch,
           path=RESULTS / "simple_rnn_loss_curve.png")

## 12. Step 9: Final Evaluation on the Test Set (Run Once!)
All hyperparameters were fixed on the validation set. We now evaluate the model **once** on the 160 unseen test reviews.

In [ ]:
probs = model.predict(data.X_test, verbose=0).ravel()
test_metrics = compute_metrics(data.y_test, probs)
ci = bootstrap_ci(data.y_test, probs)
trainable = n_params(model.trainable_weights)

print("=" * 70)
print(f"TEST RESULTS (Threshold = {THRESHOLD}):")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}  (95% CI: {ci['accuracy'][0]:.3f} - {ci['accuracy'][1]:.3f})")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1-Score:  {test_metrics['f1']:.4f}  (95% CI: {ci['f1'][0]:.3f} - {ci['f1'][1]:.3f})")
print(f"  ROC-AUC:   {test_metrics['roc_auc']:.4f}  (95% CI: {ci['roc_auc'][0]:.3f} - {ci['roc_auc'][1]:.3f})")
print(f"Confusion Matrix: TN={test_metrics['tn']}  FP={test_metrics['fp']}  FN={test_metrics['fn']}  TP={test_metrics['tp']}")
print("=" * 70)

save_result(RESULTS / "simple_rnn_metrics.json", "Simple RNN", test_metrics,
            training_time_sec=train_time, trainable_params=trainable, total_params=model.count_params(),
            epochs_run=epochs_run, best_epoch=best_epoch, extra={"bootstrap_95ci": ci})

pred_df = data.test_df[["row_id", "label", "hotel", "deceptive"]].copy()
pred_df["prob_positive"] = probs
pred_df["prediction"] = (probs >= THRESHOLD).astype(int)
pred_df.to_csv(RESULTS / "simple_rnn_test_predictions.csv", index=False)

cm = np.array([[test_metrics['tn'], test_metrics['fp']], [test_metrics['fn'], test_metrics['tp']]])
plot_confusion_matrix(cm, "Simple RNN", path=RESULTS / "simple_rnn_confusion_matrix.png")
plot_roc(data.y_test, probs, test_metrics['roc_auc'], "Simple RNN", threshold=THRESHOLD, path=RESULTS / "simple_rnn_roc_curve.png")
plot_probability_hist(data.y_test, probs, "Simple RNN", threshold=THRESHOLD, path=RESULTS / "simple_rnn_probability_hist.png")

## 13. Step 10: Error Analysis & Misclassification Inspection

In [ ]:
err = data.test_df.copy()
err["prob"] = probs
err["pred"] = (probs >= THRESHOLD).astype(int)
err["correct"] = (err["pred"] == err["label"])
err["confidence"] = np.where(err["pred"] == 1, err["prob"], 1 - err["prob"])
err["n_tokens"] = err["tokens"].map(len)
err["outcome"] = np.where(err["correct"], "CORRECT",
                          np.where((err["label"] == 0) & (err["pred"] == 1), "FP", "FN"))

print(f"Total misclassifications: {(~err.correct).sum()} / {len(err)} ({100 * (~err.correct).mean():.1f}%)")
print(f"  False Positives: {(err.outcome == 'FP').sum()} | False Negatives: {(err.outcome == 'FN').sum()}")
print(f"  Mean confidence - Correct: {err[err.correct].confidence.mean():.3f} | Wrong: {err[~err.correct].confidence.mean():.3f}")

def show_errors(outcome, k=4):
    subset = err[err.outcome == outcome].sort_values("confidence", ascending=False).head(k)
    name = "FALSE POSITIVES" if outcome == "FP" else "FALSE NEGATIVES"
    print("=" * 80)
    print(f"{name} (Top {len(subset)})")
    print("=" * 80)
    for _, r in subset.iterrows():
        print(f"[{r.hotel} | P(pos)={r.prob:.3f} | {r.n_tokens} tokens]")
        print(textwrap.fill(r.text[:350], 80) + ("..." if len(r.text) > 350 else ""))
        print("-" * 80)

show_errors("FP")
show_errors("FN")

## 14. Step 11: Multi-Seed Robustness Validation

In [ ]:
runs = [{
    "seed": SEED, "epochs": epochs_run,
    "val_acc": history["val_accuracy"][best_epoch - 1],
    "test_acc": test_metrics["accuracy"], "test_f1": test_metrics["f1"], "test_auc": test_metrics["roc_auc"]
}]

test_seeds = [1, 2, 3, 4, 5]
print("Running robustness evaluation across seeds:", test_seeds)
for s in test_seeds:
    m, h, _ = train_simple_rnn(s, verbose=0)
    p = m.predict(data.X_test, verbose=0).ravel()
    t = compute_metrics(data.y_test, p)
    b_idx = int(np.argmin(h["val_loss"]))
    runs.append({
        "seed": s, "epochs": len(h["loss"]),
        "val_acc": h["val_accuracy"][b_idx],
        "test_acc": t["accuracy"], "test_f1": t["f1"], "test_auc": t["roc_auc"]
    })
    keras.backend.clear_session()

seed_df = pd.DataFrame(runs)
print("\nMulti-Seed Stability:")
print(seed_df.round(4).to_string(index=False))
print(f"\nMean ± Std (over {len(seed_df)} seeds):")
for col in ["val_acc", "test_acc", "test_f1", "test_auc"]:
    print(f"  {col:10s}: {seed_df[col].mean():.4f} ± {seed_df[col].std():.4f}")

seed_df.to_csv(RESULTS / "simple_rnn_seed_robustness.csv", index=False)
print("Saved simple_rnn_seed_robustness.csv")

## 15. Step 12: Handover Summary for Group Report

In [ ]:
print("=" * 80)
print("ROW FOR GROUP COMPARISON TABLE:")
print("=" * 80)
print("| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC | Training Time | Parameters |")
print("|---|---|---|---|---|---|---|---|")
print(f"| Simple RNN | {test_metrics['accuracy']:.4f} | {test_metrics['precision']:.4f} | {test_metrics['recall']:.4f} | "
      f"{test_metrics['f1']:.4f} | {test_metrics['roc_auc']:.4f} | {train_time:.1f} s | {trainable:,} |")
print("=" * 80)

print(f"\nFiles in {RESULTS}:")
for p in sorted(RESULTS.iterdir()):
    if p.name != ".gitkeep":
        print(f"  {p.name:<35} ({p.stat().st_size:,} bytes)")

## 16. Step 13: Pre-Loaded Custom Review Test Battery
We evaluate a curated battery of 8 realistic hotel review test cases representing diverse sentiment challenges.

In [ ]:
test_cases = [
    ("Strong Positive", "Absolutely breathtaking hotel! The rooms were immaculately clean, beds exceptionally comfortable, and the staff treated us like royalty."),
    ("Strong Negative", "Terrible experience from start to finish. The room smelled like damp mildew, the carpet had filthy stains, and the receptionist was exceedingly rude."),
    ("Tricky Negation 1", "The room was not clean at all, and I certainly would not recommend this place to anyone visiting Chicago."),
    ("Tricky Negation 2", "No complaints whatsoever! Nothing was broken, no delays at check-in, and the service was not disappointing in any way."),
    ("Subtle Disappointment", "The photos on their official website looked gorgeous, but the actual room was tiny, poorly lit, and facing a noisy alleyway."),
    ("Short Positive", "Wonderful stay, loved every minute!"),
    ("Short Negative", "Horrible, never going back."),
    ("Mixed Review", "The breakfast buffet was delicious and staff were very courteous, but our air conditioner rattled loudly all night preventing any real sleep.")
]

def predict_single_review(text_input, trained_model, vocab_dict):
    cleaned_str = clean_text(text_input)
    token_list = cleaned_str.split()
    encoded_ids = encode(token_list, vocab_dict)
    padded_seq = pad([encoded_ids], max_len=MAX_LEN)
    prob_pos = float(trained_model.predict(padded_seq, verbose=0)[0][0])
    sentiment = "POSITIVE" if prob_pos >= THRESHOLD else "NEGATIVE"
    confidence = prob_pos if sentiment == "POSITIVE" else 1.0 - prob_pos
    return sentiment, prob_pos, confidence, cleaned_str

print(f"{'Category':<24} | {'Sentiment':<8} | {'P(Pos)':<6} | {'Conf':<5} | {'Cleaned Excerpt'}")
print("-" * 80)
for category, sample_text in test_cases:
    sent, p_pos, conf, cleaned_t = predict_single_review(sample_text, model, data.vocab)
    print(f"{category:<24} | {sent:<8} | {p_pos:.4f} | {conf:.1%} | {cleaned_t[:30]}...")

## 17. Step 14: Interactive Review Classification
Type any custom hotel review below to classify it with the trained Simple RNN model.

In [ ]:
user_review = input("Enter a hotel review to test (or press Enter for default): ").strip()
if not user_review:
    user_review = "The hotel room was not clean, the shower was freezing cold, but the breakfast was quite decent."

sent, p_pos, conf, cleaned_t = predict_single_review(user_review, model, data.vocab)
print("\n" + "=" * 60)
print("INTERACTIVE PREDICTION RESULT")
print("=" * 60)
print(f"Input Review:     {user_review}")
print(f"Cleaned Tokens:   {cleaned_t}")
print(f"Predicted Class:  {sent}")
print(f"P(Positive):      {p_pos:.4f}")
print(f"Confidence:       {conf:.2%}")
print("=" * 60)